# Patch-level Anomaly Detection + Defect-Type Classification (실험)

**폴더 하나(클래스별 하위폴더)를 받아 support/query 로 자동 분리**해 오염 없이 테스트한다.

- 정상 클래스(`NORMAL_CLASS`)의 support → **정상 patch 메모리**. 나머지 = 불량 클래스.
- 불량 support 에서 **정상 대비 이질적인 patch만** 골라 **불량종류별 bank** 구성.
- **query(=bank 에 없는 이미지)** 로만 시각화(§6)·정량평가(§7) → 정직한 결과.

> 서버(GPU), 백본 직접. feature 는 한 번만 추출해 캐시 → seed 별 재분할이 빠름.

## 0. 경로 / 임포트

In [ ]:
import os, sys
from pathlib import Path
_HERE = Path.cwd(); _REPO = _HERE
while not (_REPO / 'dino_v3').exists() and _REPO != _REPO.parent:
    _REPO = _REPO.parent
for p in (str(_REPO), str(_REPO / 'dino_v3')):
    if p not in sys.path:
        sys.path.insert(0, p)
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import dinov3.distributed as distributed
from dinov3.eval.setup import setup_and_build_model
from inspection.em_aug import build_em_eval_transform
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. 설정
`DATA_ROOT` = 클래스별 폴더(classA/ classB/ ...). 그중 정상 폴더명을 `NORMAL_CLASS` 로 지정.
정상 폴더가 없으면(`NORMAL_CLASS` 매칭 안 되면) 정상-대조 없이 순수 patch 분류로 폴백.

In [ ]:
CONFIG      = 'dino_v3/dinov3/configs/train/weaksup/stage2_ssl_weaksup.yaml'  # FILL_IN
CKPT        = '/path/to/teacher_checkpoint.pth'   # FILL_IN
DATA_ROOT   = '/path/to/class_folders'            # FILL_IN (classA/ classB/ ...)
NORMAL_CLASS= 'normal'     # 정상 폴더명 (없으면 폴백)
SUPPORT_N   = 10           # 클래스당 support(bank) 이미지 수, 나머지는 query
N_SEEDS     = 3            # 정량평가 반복(평균±std). 시각화는 seed 0
IMAGE_SIZE  = 448
N_BLOCKS    = 1
MEM_CAP     = 20000        # 정상 patch 메모리 최대(subsample)
ANOM_PCTL   = 99.0         # 정상 query patch 이상도의 몇 퍼센타일을 임계로
MIN_ANOM_FRAC = 0.003      # 이미지 판정: 이상 patch 비율 임계
MAX_IMG_PER_CLASS = 60
CONFIG = str((_REPO / CONFIG)) if not os.path.isabs(CONFIG) else CONFIG

## 2. 백본 로드 + patch feature 추출기

In [ ]:
if not distributed.is_enabled():
    distributed.enable(overwrite=True)
model, ctx = setup_and_build_model(config_file=CONFIG, pretrained_weights=CKPT, output_dir='./.em_cache')
model.cuda().eval()
autocast_dtype = ctx['autocast_dtype']
eval_tf = build_em_eval_transform(IMAGE_SIZE)

@torch.inference_mode()
def patch_features(pil):
    x = eval_tf(pil.convert('RGB')).unsqueeze(0).cuda()
    with torch.autocast('cuda', dtype=autocast_dtype):
        outs = model.get_intermediate_layers(x, n=N_BLOCKS, reshape=True, return_class_token=True, norm=True)
    grid = outs[-1][0][0].float().cpu().numpy()          # [C,h,w]
    C, h, w = grid.shape
    f = grid.reshape(C, h * w).T
    f = f / (np.linalg.norm(f, axis=1, keepdims=True) + 1e-8)
    return f.astype(np.float32), h, w

IMG_EXTS = {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

## 3. 모든 이미지 feature 한 번만 추출 (캐시)
이후 support/query 분할·bank 구성은 numpy 로만 → seed 반복이 빠름.

In [ ]:
root = Path(DATA_ROOT)
class_dirs = sorted([d for d in root.iterdir() if d.is_dir()])
assert class_dirs, f'클래스 폴더 없음: {root}'
data = {}   # class -> list of dict(path, feats[P,D], h, w)
for d in class_dirs:
    ps = [p for p in sorted(d.rglob('*')) if p.suffix.lower() in IMG_EXTS][:MAX_IMG_PER_CLASS]
    lst = []
    for p in ps:
        f, h, w = patch_features(Image.open(p))
        lst.append(dict(path=str(p), feats=f, h=h, w=w))
    data[d.name] = lst
    print(f'  {d.name}: {len(lst)}장')

HAS_NORMAL = NORMAL_CLASS in data
defect_names = [c for c in data if c != NORMAL_CLASS]
print('HAS_NORMAL:', HAS_NORMAL, '| defect classes:', defect_names)

## 4. split → bank 구성 함수 (numpy)
seed 마다 클래스별 support/query 분할. 정상 support=메모리, 정상 query=임계 보정.
불량 support 에서 이상 patch만 골라 bank. **모두 query 는 bank 에 안 들어감.**

In [ ]:
def split_sq(n, k, rng):
    idx = rng.permutation(n)
    k = min(k, max(1, n // 2))
    return idx[:k], idx[k:]

def stack(items):
    return np.vstack([it['feats'] for it in items]) if items else np.zeros((0, model.embed_dim), np.float32)

def build(seed):
    rng = np.random.default_rng(seed)
    sup, qry = {}, {}
    for c, lst in data.items():
        s, q = split_sq(len(lst), SUPPORT_N, rng)
        sup[c] = [lst[i] for i in s]; qry[c] = [lst[i] for i in q]

    if HAS_NORMAL:
        M = stack(sup[NORMAL_CLASS])
        if len(M) > MEM_CAP:
            M = M[rng.choice(len(M), MEM_CAP, replace=False)]
        def anom(qf):
            out = np.empty(len(qf), np.float32)
            for i in range(0, len(qf), 4096):
                out[i:i+4096] = 1.0 - (qf[i:i+4096] @ M.T).max(axis=1)
            return out
        nq = stack(qry[NORMAL_CLASS])
        thr = float(np.percentile(anom(nq), ANOM_PCTL)) if len(nq) else 0.5
        banks = []
        for c in defect_names:
            f = stack(sup[c]); a = anom(f); banks.append((c, f[a > thr]))
    else:  # 폴백: 정상 없음 → 모든 클래스 bank, 이상 gate 없음
        M, anom, thr = None, None, None
        banks = [(c, stack(sup[c])) for c in data]
    return dict(sup=sup, qry=qry, M=M, anom=anom, thr=thr, banks=banks)

def classify_patches(qf, banks):
    best = np.full(len(qf), -1, int); bestsim = np.full(len(qf), -1.0, np.float32)
    for ci, (_, arr) in enumerate(banks):
        if len(arr) == 0: continue
        s = (qf @ arr.T).max(axis=1)
        m = s > bestsim; best[m] = ci; bestsim[m] = s[m]
    return best, bestsim

def analyze(item, B):
    f = item['feats']; h, w = item['h'], item['w']
    banks = B['banks']; names = [nm for nm, _ in banks]
    if HAS_NORMAL:
        a = B['anom'](f); is_anom = a > B['thr']
        ttype, tsim = classify_patches(f, banks)
        type_map = np.where(is_anom, ttype, -1)
        if is_anom.mean() < MIN_ANOM_FRAC:
            pred = NORMAL_CLASS
        else:
            mass = {nm: float(tsim[is_anom & (ttype == ci)].sum()) for ci, nm in enumerate(names)}
            pred = max(mass, key=mass.get) if any(v > 0 for v in mass.values()) else NORMAL_CLASS
        return dict(h=h, w=w, anom=a, is_anom=is_anom, type_map=type_map, pred=pred, names=names)
    else:
        ttype, tsim = classify_patches(f, banks)
        mass = {nm: float(tsim[ttype == ci].sum()) for ci, nm in enumerate(names)}
        return dict(h=h, w=w, anom=None, is_anom=None, type_map=ttype, pred=max(mass, key=mass.get), names=names)

## 5. 정상 대조 필터 점검 (bank 등록 비율)

In [ ]:
B0 = build(0)
if HAS_NORMAL:
    print(f'이상 임계 thr = {B0["thr"]:.4f}')
    for c, arr in B0['banks']:
        tot = len(stack(B0['sup'][c]))
        print(f'  {c}: support patch {tot} → 이상 patch {len(arr)} ({100*len(arr)/max(1,tot):.1f}%) bank')
else:
    print('정상 클래스 없음 → 순수 patch 분류(폴백). 이상 gate 비활성.')

## 6. 시각화 — **query 이미지**로 (bank 에 없는 이미지, 정직)
입력 | anomaly heatmap | 불량종류 overlay. 각 클래스 query 에서 몇 장.

In [ ]:
CLASS_COLORS = plt.cm.tab10(np.arange(len(B0['banks'])) % 10)[:, :3]

def show(item, r):
    h, w = r['h'], r['w']; names = r['names']
    pil = Image.open(item['path'])
    base = np.asarray(pil.resize((IMAGE_SIZE, IMAGE_SIZE)).convert('L'), float) / 255
    tm = r['type_map'].reshape(h, w)
    over = np.zeros((h, w, 4), np.float32)
    for ci in range(len(names)):
        over[tm == ci, :3] = CLASS_COLORS[ci]; over[tm == ci, 3] = 0.8
    over_img = np.asarray(Image.fromarray((over*255).astype(np.uint8)).resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)).astype(float)/255
    ncol = 3 if r['anom'] is not None else 2
    fig, ax = plt.subplots(1, ncol, figsize=(4.3*ncol, 4.3))
    ax[0].imshow(base, cmap='gray'); ax[0].set_title(f'input\npred={r["pred"]}'); ax[0].axis('off')
    j = 1
    if r['anom'] is not None:
        am = np.asarray(Image.fromarray(r['anom'].reshape(h, w).astype(np.float32)).resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR))
        im = ax[1].imshow(am, cmap='inferno'); ax[1].set_title(f'anomaly (thr={r_thr:.3f})'); ax[1].axis('off')
        fig.colorbar(im, ax=ax[1], fraction=0.046); j = 2
    ax[j].imshow(base, cmap='gray'); ax[j].imshow(over_img); ax[j].set_title('defect type'); ax[j].axis('off')
    handles = [plt.Line2D([0],[0], marker='s', ls='', color=CLASS_COLORS[ci], label=names[ci]) for ci in range(len(names))]
    ax[j].legend(handles=handles, fontsize=7, loc='upper right')
    plt.tight_layout(); plt.show()

r_thr = B0['thr'] if HAS_NORMAL else 0.0
for c in data:
    for item in B0['qry'][c][:2]:
        show(item, analyze(item, B0))

## 7. 정량 — **query only**, seed 반복 (평균±std)
bank 에 없는 query 이미지로만 이미지 단위 종류 정확도 + 혼동행렬.

In [ ]:
all_names = list(data.keys())
accs = []
conf = np.zeros((len(all_names), len(all_names)), int)   # true x pred
for seed in range(N_SEEDS):
    B = build(seed)
    ok = tot = 0
    for c in data:
        ci = all_names.index(c)
        for item in B['qry'][c]:
            pred = analyze(item, B)['pred']
            pj = all_names.index(pred) if pred in all_names else ci  # pred 는 항상 클래스명
            conf[ci, pj] += 1
            ok += int(pred == c); tot += 1
    accs.append(ok / max(1, tot))
    print(f'  seed {seed}: query acc = {accs[-1]*100:.1f}%  (n={tot})')
print(f'\nquery 종류 정확도: {np.mean(accs)*100:.1f}% ± {np.std(accs)*100:.1f}%  (seed {N_SEEDS}회)')

fig, ax = plt.subplots(figsize=(1.1*len(all_names)+2, 1.1*len(all_names)+1))
ax.imshow(conf, cmap='Blues')
ax.set_xticks(range(len(all_names))); ax.set_xticklabels(all_names, rotation=45, ha='right')
ax.set_yticks(range(len(all_names))); ax.set_yticklabels(all_names)
ax.set_xlabel('pred'); ax.set_ylabel('true'); ax.set_title('confusion (query, 누적)')
for i in range(len(all_names)):
    for j in range(len(all_names)): ax.text(j, i, conf[i, j], ha='center', va='center')
plt.tight_layout(); plt.show()

## 참고
- **query 는 bank 에 없는 이미지** → 정직한 평가(오염 없음).
- 정상 query patch 로 임계를 잡으므로, 정상 query 는 임계 보정 겸 정상 판정에 쓰임(경미한 이중사용).
- 튜닝: `SUPPORT_N`(few-shot 수), `ANOM_PCTL`(이상 임계), `MIN_ANOM_FRAC`(이미지 판정),
  `N_BLOCKS`(레이어; 중간층 실험은 `get_intermediate_layers(n=[3,6,9])`).